---

# 🧠 Week 2 — Neural Network Experiments

A Multi-Layer Perceptron (MLP) is used as the experimental model.

Three experiments are performed to investigate how changes in network
architecture and activation function affect ASL classification performance.

| Experiment | Hidden Layers | Activation |
|---|---|---|
| Experiment 1 | 128 | ReLU |
| Experiment 2 | 256 → 128 | ReLU |
| Experiment 3 | 256 → 128 | Tanh |

All experiments are tracked using **Weights & Biases (W&B)**.

# 7. PyTorch Data Preparation

Before training the Multi-Layer Perceptron (MLP), the normalized image
data must be converted into a format that PyTorch can use.

The data preparation process includes:

1. Converting the pandas data into NumPy arrays.
2. Mapping the class labels to consecutive values.
3. Converting the data into PyTorch tensors.
4. Pairing images with their correct labels using TensorDataset.
5. Creating DataLoaders that provide the neural network with batches of data.

In [11]:
# Convert pandas data into NumPy arrays

X_train_np = X_train.to_numpy(dtype=np.float32)
X_test_np = X_test.to_numpy(dtype=np.float32)

y_train_np = y_train.to_numpy(dtype=np.int64)
y_test_np = y_test.to_numpy(dtype=np.int64)

print("Training images:", X_train_np.shape)
print("Testing images:", X_test_np.shape)
print("Training labels:", y_train_np.shape)
print("Testing labels:", y_test_np.shape)

Training images: (27455, 784)
Testing images: (7172, 784)
Training labels: (27455,)
Testing labels: (7172,)


In [13]:
# Map the original labels to consecutive class numbers

unique_labels = sorted(np.unique(y_train_np))

label_to_index = {
    label: index
    for index, label in enumerate(unique_labels)
}

index_to_label = {
    index: label
    for label, index in label_to_index.items()
}

# Apply the new label mapping
y_train_mapped = np.array([
    label_to_index[label]
    for label in y_train_np
])

y_test_mapped = np.array([
    label_to_index[label]
    for label in y_test_np
])

num_classes = len(unique_labels)

print("Original labels:", unique_labels)
print("Number of classes:", num_classes)
print("Mapped labels:", sorted(np.unique(y_train_mapped)))

Original labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Number of classes: 24
Mapped labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]


In [14]:
# Convert NumPy arrays into PyTorch tensors

X_train_tensor = torch.tensor(
    X_train_np,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_np,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_mapped,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test_mapped,
    dtype=torch.long
)

print("Training images:", X_train_tensor.shape)
print("Training labels:", y_train_tensor.shape)
print("Testing images:", X_test_tensor.shape)
print("Testing labels:", y_test_tensor.shape)

Training images: torch.Size([27455, 784])
Training labels: torch.Size([27455])
Testing images: torch.Size([7172, 784])
Testing labels: torch.Size([7172])


In [15]:
# Pair images with labels and create DataLoaders

batch_size = 64

# Pair each image with its correct label
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create batches for training and testing
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Batch size:", batch_size)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Batch size: 64
Training batches: 429
Testing batches: 113


# 8. Experiment 1 — Single Hidden Layer + ReLU

### Architecture

**784 inputs → 128 hidden neurons → ReLU → 24 outputs**

This experiment establishes the initial neural-network baseline.

## 8.1 Model Architecture

Experiment 1 uses a Multi-Layer Perceptron with one hidden layer
containing 128 neurons and the ReLU activation function.

**Architecture:** 784 Inputs → 128 Hidden Neurons → ReLU → 24 Outputs

In [16]:
# Experiment 1: Single Hidden Layer + ReLU

class MLP1(nn.Module):

    def __init__(self):
        super(MLP1, self).__init__()

        self.network = nn.Sequential(
            # Input layer: 784 pixels → 128 neurons
            nn.Linear(784, 128),

            # Activation function
            nn.ReLU(),

            # Output layer: 128 neurons → 24 ASL classes
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


# Select GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Create the model
model_1 = MLP1().to(device)

print("Using device:", device)
print(model_1)

Using device: cpu
MLP1(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=24, bias=True)
  )
)


## 8.2 Experiment Tracking

Weights & Biases (W&B) is used to record the training loss,
training accuracy, test accuracy, and experiment configuration.

In [17]:
# Start Experiment 1 in Weights & Biases

wandb.init(
    project="asl-translation-engine",
    name="MLP_1_128_ReLU",
    config={
        "hidden_layers": 1,
        "hidden_neurons": 128,
        "activation": "ReLU",
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 10
    }
)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer_1 = optim.Adam(
    model_1.parameters(),
    lr=0.001
)

print("Experiment 1 is ready!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: baileyae (baileyae-university-of-louisiana-monroe) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Experiment 1 is ready!


## 8.3 Model Training

The model is trained for 10 epochs using batches of 64 images.
Training loss and accuracy are recorded after each epoch and sent
to W&B for experiment tracking.

In [18]:
# Train Experiment 1

epochs = 10

for epoch in range(epochs):

    model_1.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Move batch to GPU/CPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients from previous step
        optimizer_1.zero_grad()

        # Forward pass
        outputs = model_1(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer_1.step()

        # Track loss
        total_loss += loss.item()

        # Determine predicted classes
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate results for this epoch
    average_loss = total_loss / len(train_loader)
    training_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {average_loss:.4f} | "
        f"Accuracy: {training_accuracy:.4f}"
    )

    # Send results to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": average_loss,
        "train_accuracy": training_accuracy
    })

Epoch 1/10 | Loss: 2.5303 | Accuracy: 0.2665
Epoch 2/10 | Loss: 1.6447 | Accuracy: 0.5136
Epoch 3/10 | Loss: 1.3004 | Accuracy: 0.6142
Epoch 4/10 | Loss: 1.1022 | Accuracy: 0.6696
Epoch 5/10 | Loss: 0.9651 | Accuracy: 0.7121
Epoch 6/10 | Loss: 0.8576 | Accuracy: 0.7458
Epoch 7/10 | Loss: 0.7768 | Accuracy: 0.7699
Epoch 8/10 | Loss: 0.6983 | Accuracy: 0.7973
Epoch 9/10 | Loss: 0.6392 | Accuracy: 0.8193
Epoch 10/10 | Loss: 0.5878 | Accuracy: 0.8332


## 8.4 Experiment 1 Evaluation

After training, the model is evaluated using the test dataset.
These images were not used to update the model during training,
making test accuracy a better measure of how well the model
generalizes to unseen data.

In [19]:
# Evaluate Experiment 1

model_1.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Make predictions
        outputs = model_1(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


# Calculate test accuracy
test_accuracy_1 = correct / total

print(
    f"Experiment 1 Test Accuracy: "
    f"{test_accuracy_1 * 100:.2f}%"
)

# Save final result to W&B
wandb.log({
    "test_accuracy": test_accuracy_1
})

# Finish this W&B run
wandb.finish()

Experiment 1 Test Accuracy: 65.10%


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▄▃▂▂▂▁▁▁
epoch,10
test_accuracy,0.651
train_accuracy,0.83318
train_loss,0.58777


### Experiment 1 Results

**Architecture:** 784 → 128 → 24  
**Activation Function:** ReLU  
**Epochs:** 10  
**Batch Size:** 64  
**Learning Rate:** 0.001  
**Test Accuracy:** approximately 66.72%

Experiment 1 establishes the initial neural-network baseline.
The next experiment increases the size and depth of the network
to determine whether a more complex architecture improves
classification performance.